## Setup Instructions

**Before running the ingestion cell**, you need to:

1. **Export your SQL Server tables to CSV files:**
   - Open SQL Server Management Studio
   - Right-click on each table (names, people, RawJsonLog)
   - Select "Tasks" → "Export Data" → Choose "Flat File Destination"
   - Save as: `names.csv`, `people.csv`, `RawJsonLog.csv`

2. **Create the Unity Catalog Volume:**
   ```sql
   CREATE VOLUME IF NOT EXISTS cricketscoreprediction.bronze.raw_data;
   ```

3. **Upload CSV files to the Volume:**
   - In Databricks UI, navigate to **Catalog** → **cricketscoreprediction** → **bronze** → **raw_data**
   - Click "Upload" and select your CSV files

4. **Run the ingestion cell** to load data into Bronze Delta tables

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS cricketscoreprediction.bronze
COMMENT 'Bronze layer: Raw data'
""")
print("✓ Bronze schema created")

In [0]:
ipl_json_path = "/Volumes/cricketscoreprediction/bronze/raw_data/ipl_json/"
names_path = "/Volumes/cricketscoreprediction/bronze/raw_data/names/"
people_path = "/Volumes/cricketscoreprediction/bronze/raw_data/people/"

# Read and store IPL matches data
matches_df = spark.read.option("multiLine", "true").json(ipl_json_path)
matches_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable("cricketscoreprediction.bronze.matchesdata")
print(f"✓ Created table: cricketscoreprediction.bronze.matches ({matches_df.count()} rows)")



# Read and store names data (trying common formats)
try:
    names_df = spark.read.option("header", "true").option("inferSchema", "true").csv(names_path)
except:
    try:
        names_df = spark.read.json(names_path)
    except:
        names_df = spark.read.parquet(names_path)

names_df.write.mode("overwrite").saveAsTable("cricketscoreprediction.bronze.names")
print(f"✓ Created table: cricketscoreprediction.bronze.names ({names_df.count()} rows)")

# Read and store people data (trying common formats)
try:
    people_df = spark.read.option("header", "true").option("inferSchema", "true").csv(people_path)
except:
    try:
        people_df = spark.read.json(people_path)
    except:
        people_df = spark.read.parquet(people_path)

people_df.write.mode("overwrite").saveAsTable("cricketscoreprediction.bronze.people")
print(f"✓ Created table: cricketscoreprediction.bronze.people ({people_df.count()} rows)")

print("\n✅ All bronze tables created successfully!")